# 05 - 将财报章节切成文本块

上一节已经提取出 `Business`、`Risk Factors` 和 `MD&A`。这一节使用 LangChain 将长章节切成适合大模型逐块分析的小文本，同时保留每一块的原文位置。

```text
标准章节 JSON -> RecursiveCharacterTextSplitter -> 带来源位置的 chunks JSON
```

## 1. 读取章节 JSON

Notebook 自动读取指定公司最新的 `_sections.json` 文件。

In [ ]:
import json
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter

TICKER = "AAPL"
FORM_TYPE = "10-K"
DATA_DIR = Path("data/sec")

section_files = sorted(
    (DATA_DIR / TICKER).glob(f"*_{FORM_TYPE}_*_sections.json"),
    reverse=True,
)
if not section_files:
    raise FileNotFoundError(
        f"没有找到 {TICKER} 的章节 JSON，请先运行 04_extract_sections.ipynb"
    )

sections_path = section_files[0]
filing_data = json.loads(sections_path.read_text(encoding="utf-8"))
sections = filing_data["sections"]

print(f"读取文件：{sections_path}")
for name, section in sections.items():
    print(f"{name:>12}: {section['char_count']:>7,} 字符")

## 2. 配置 LangChain 文本切分器

`chunk_size=4000` 表示每块最多约 4,000 个字符。`chunk_overlap=400` 让相邻块保留部分重复内容，降低句子或段落在边界处被截断的影响。

这里使用字符数而不是 Token 数，便于观察和学习。后续可以改成按模型 Token 计数。

In [ ]:
CHUNK_SIZE = 4000
CHUNK_OVERLAP = 400

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
    add_start_index=True,
)

print(f"最大块长度：{CHUNK_SIZE} 字符")
print(f"目标重叠长度：{CHUNK_OVERLAP} 字符")

## 3. 切分单个章节并添加元数据

LangChain 返回每个文本块在章节内的 `start_index`。再加上上一节记录的章节起点，就能得到它在完整财报文本中的绝对位置。

In [ ]:
def chunk_section(
    ticker: str,
    form_type: str,
    section_name: str,
    section: dict,
) -> list[dict]:
    documents = splitter.create_documents(
        texts=[section["text"]],
        metadatas=[{
            "ticker": ticker,
            "form": form_type,
            "section": section_name,
            "section_title": section["title"],
        }],
    )

    chunks = []
    total_chunks = len(documents)

    for index, document in enumerate(documents, start=1):
        local_start = document.metadata["start_index"]
        local_end = local_start + len(document.page_content)
        source_start = section["start"] + local_start
        source_end = section["start"] + local_end

        chunks.append({
            "chunk_id": f"{ticker}_{section_name}_{index:03d}",
            "ticker": ticker,
            "form": form_type,
            "section": section_name,
            "section_title": section["title"],
            "chunk_index": index,
            "total_chunks": total_chunks,
            "local_start": local_start,
            "local_end": local_end,
            "source_start": source_start,
            "source_end": source_end,
            "char_count": len(document.page_content),
            "text": document.page_content,
        })

    return chunks

## 4. 切分全部章节

三个章节使用同一套参数和同一套输出结构。

In [ ]:
all_chunks = []

for section_name, section in sections.items():
    section_chunks = chunk_section(
        ticker=filing_data["ticker"],
        form_type=filing_data["form"],
        section_name=section_name,
        section=section,
    )
    all_chunks.extend(section_chunks)

    sizes = [chunk["char_count"] for chunk in section_chunks]
    print(
        f"{section_name:>12}: {len(section_chunks):>2} 块 | "
        f"最短 {min(sizes):,} | 最长 {max(sizes):,} | "
        f"平均 {sum(sizes) / len(sizes):,.0f}"
    )

print(f"\n总计：{len(all_chunks)} 个文本块")

## 5. 查看一个文本块

这里同时显示内容和来源位置。以后 DeepSeek 对这一块生成结论时，可以把 `chunk_id` 作为引用编号。

In [ ]:
example_chunk = all_chunks[0]

print(f"chunk_id：{example_chunk['chunk_id']}")
print(f"章节：{example_chunk['section_title']}")
print(
    f"章节内位置：{example_chunk['local_start']:,} - "
    f"{example_chunk['local_end']:,}"
)
print(
    f"完整财报位置：{example_chunk['source_start']:,} - "
    f"{example_chunk['source_end']:,}"
)
print(f"字符数：{example_chunk['char_count']:,}\n")
print(example_chunk["text"][:1000])

## 6. 观察相邻块的重叠

`400` 是目标重叠上限。切分器会优先在段落、换行和单词边界处分割，所以实际重叠长度可能小于 400。

In [ ]:
business_chunks = [
    chunk for chunk in all_chunks if chunk["section"] == "business"
]

for previous, current in zip(business_chunks, business_chunks[1:]):
    overlap = previous["local_end"] - current["local_start"]
    print(
        f"{previous['chunk_id']} -> {current['chunk_id']}: "
        f"重叠 {max(0, overlap)} 字符"
    )

## 7. 验证切块结果

校验每块不为空、不超过最大长度、位置递增，并确认保存的文本可以在原章节的对应位置精确找到。

In [ ]:
for section_name, section in sections.items():
    section_chunks = [
        chunk for chunk in all_chunks if chunk["section"] == section_name
    ]
    starts = [chunk["local_start"] for chunk in section_chunks]
    assert starts == sorted(starts), f"{section_name} 的文本块顺序异常"

    for chunk in section_chunks:
        assert chunk["text"], f"{chunk['chunk_id']} 是空文本"
        assert chunk["char_count"] <= CHUNK_SIZE
        assert (
            section["text"][chunk["local_start"]:chunk["local_end"]]
            == chunk["text"]
        ), f"{chunk['chunk_id']} 无法对应到章节原文"

print("文本块长度、顺序和原文位置检查通过")

## 8. 保存标准化 chunks JSON

输出文件包含切分配置、原章节文件和全部文本块，后续 DeepSeek 分析可以直接读取。

In [ ]:
output_data = {
    "ticker": filing_data["ticker"],
    "form": filing_data["form"],
    "source_sections_file": str(sections_path),
    "chunk_config": {
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "length_unit": "characters",
    },
    "chunks": all_chunks,
}

output_path = sections_path.with_name(
    sections_path.name.replace("_sections.json", "_chunks.json")
)
output_path.write_text(
    json.dumps(output_data, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"文本块文件已保存：{output_path.resolve()}")
print(f"共保存 {len(all_chunks)} 个文本块")

## 这一节完成了什么

现在三个长章节已经被转换成统一、可追溯的小文本块。每个块都有稳定的 `chunk_id`、所属章节和原文位置。

下一步是第一次正式调用 DeepSeek：分别分析每个 chunk，输出结构化事实和引用编号，再按章节汇总。